<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_Simulation_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

## Middle East Oil Security Simulation Engine
MOS is an LLM-driven geopolitical simulation in which strategic actors choose actions within an explicitly represented and evolving WorldState. Chanakya proposes actor-specific strategic options, Centaur evaluates their relative likelihood, and ZeitWorld translates the selected Event into the next WorldState. The objective is not to predict the future, but to explore plausible, path-dependent trajectories arising from alternative strategic choices.

In [1]:
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mos2.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos2.py


import mos
import mos2

from mos import *
from mos2 import *

In [ ]:
import importlib


importlib.reload(mos)
importlib.reload(mos2)

from mos import *
from mos2 import *

In [2]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


## WorldState and ZeitWorld

In [3]:
# These are the eight variables that currently define the World
# This number can be increased if necessasary
#
# ---------------------------------------------------------------

worldVariables = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [4]:
# Historical Data and
# LLM Roles and prompts are stored and retrieved from Google Drive
#
# ----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"

!ls -l "{WORLDFACTS_DIR}"
!ls -lh "{WORLDFACTS_DIR}/MD"
!ls -lh "{WORLDFACTS_DIR}/NEWS"
!ls -lh "{ZW_PATH}"

Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF
total 159K
-rw------- 1 root root 7.7K Oct  1 09:31 IranData_261001.md
-rw------- 1 root root 6.9K Oct  1 09:31 MissilesofIranMissil_261001.md
-rw------- 1 root root 8.1K Oct  1 09:32 OilMarketReportSepte_261001.md
-rw------- 1 root root  13K Oct  1 09:32 PressReleasePagePres_261001.md
-rw------- 1 root root 4.3K Oct  1 09:33 ShortTermEnergyOutlo_261001.md
-rw------- 1 root root  31K Oct  1 09:34 SnapshotofIndiasOilG_261001.md
-rw------- 1 root root  18K Oct  1 09:35 UKCommonsIranin2026S_261001.md
-rw------- 1 root root  70K Oct  1 09:37 USCongressR488873_261001.md
total 56K
-rw------- 1 root root 56K Oct  1 09:39 MOS-MC-LiveFeed_261001_1509_IST.txt
total 13K
-rw------- 1 root root 2.2K Oct  1 00:10 ZW_Initialisation_Prompt_v0.txt
-rw------- 1 root root 3.1K Oct  1 14:08 ZW_Initialisation_Prompt_v1.txt
-rw------- 1 roo

In [5]:
# Knowledge of the initial world state is collected from a
# a set of PDF files that have been converted to markdown format
# live news data from 1 source, viz. MoneyControl.com
# Assembled into one large corpus
#
# ------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)


Static sources : 8
Live sources   : 1
Characters     : 215,849
Words          : 32,422


In [6]:
# Zeitworld Roles and Prompts are retrieved from Google Drive

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")

#ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")

ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [7]:
# ZeitWorld obect is instantiated from its class along with its ROLE
#
ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
print(ZeitWorld.initialised)
#print(ZeitWorld.model)

False


In [8]:
# W is a list of evolving World States
# An empty list is created. A blank world template is created, without data
# ----------------------------

W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

In [9]:
# ZeitWorld reads WorldFactBase data corpus and creates a World State based on World Template
# This initial World State is inserted into the World List W and becomes W[0]
#
# ----------------------------------------

#W[0] = ZeitWorld.initialise(W[0], WorldFactBase, ZW_Initialisation_Prompt)
_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 52,111 | Output: 2,119 | Reasoning: 166 | Total: 54,230
True


In [12]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

W[0].show(-2)  # ---- just the world vectors
W[0].show()    # ---- the entire state
W[0].show(4)    # ---- details of any one vector

[100, 100, 100, 100, 100, 100, 100, 100]

World State : W0
-------------------------------------------------------
 0  Oil Price                          100.00
 1  Hormuz Oil Flow                    100.00
 2  India Gulf Oil Imports             100.00
 3  Iran Sanctions Enforcement         100.00
 4  US/Allied Cohesion                 100.00
 5  Iran Internal Cohesion             100.00
 6  US-Iran Tension                    100.00
 7  Regional Armed Conflict            100.00

World State : W0

Vector 4 : US/Allied Cohesion
----------------------------------------------------------------------
Index       : 100
Description : Degree of alignment between the United States and cooperating states in their response to Iran and the Gulf crisis.
Direction   : Higher index means greater US/allied cohesion.
Status      : The United States and Israel have conducted coordinated military operations, and the UK has stated that it is working with the United States and other partners on economic pr

## Turn & Events
A turn consists of two successive events by two different actors

In [13]:
# E is a list of events that will define History
#
E = []

In [14]:
# Trigger is the event that commences the turn
#

Trigger = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    )
)

Trigger.event_id = f"E{len(E)}"
E.append(Trigger)
E[0].show()

Event  : E0
Actor  : Iran
Action : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.


In [15]:
# ZeitWorld updates the world [W], moving it from W[t] to W[t+1] using Trigger Event E[t]
#
# -------------------------------

_ = ZeitWorld.update( W, E[0], ZW_Update_Prompt)

Updating WorldState ---------- W0
Model: gpt-5.6-luna | Effort: medium | Input: 3,268 | Output: 2,584 | Reasoning: 373 | Total: 5,852
Appended  WorldState ---------- W1


In [16]:
print(len(W))
print(W[1].state_id)
W[1].show()

W[1].show(0)   # Oil Price
W[1].show(1)   # Hormuz Oil Flow
W[1].show(6)   # India Gulf Oil Imports

2
W1

World State : W1
-------------------------------------------------------
 0  Oil Price                          118.00
 1  Hormuz Oil Flow                     65.00
 2  India Gulf Oil Imports              88.00
 3  Iran Sanctions Enforcement         100.00
 4  US/Allied Cohesion                 100.00
 5  Iran Internal Cohesion             100.00
 6  US-Iran Tension                    115.00
 7  Regional Armed Conflict            114.00

World State : W1

Vector 0 : Oil Price
----------------------------------------------------------------------
Index       : 118
Description : International crude oil price level, represented by the Brent benchmark.
Direction   : Higher index means higher oil price.
Status      : Brent crude prices are subject to a further upward shock after Iran declared the Strait of Hormuz closed to commercial shipping and ordered attacks on tanker transit and US naval forces. Prices remain elevated and highly volatile amid constrained exports, reduced inventor

In [17]:
# Reaction is an event that is the response to the Trigger event
#
# -----------------------------

Reaction = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    )
)

Reaction.event_id = f"E{len(E)}"
E.append(Reaction)
E[1].show()



Event  : E1
Actor  : US
Action : The United States announces an immediate suspension of offensive military operations against Iran and offers direct negotiations on maritime security and sanctions relief, while maintaining defensive protection of US forces and commercial shipping in the Gulf.


In [18]:
# ZeitWorld updates the world [W], moving it from W[t] to W[t+1] using Trigger Event E[t]
#
# -------------------------------

_ = ZeitWorld.update( W,E[1],ZW_Update_Prompt)

Updating WorldState ---------- W1
Model: gpt-5.6-luna | Effort: medium | Input: 3,531 | Output: 2,558 | Reasoning: 210 | Total: 6,089
Appended  WorldState ---------- W2


In [19]:
print(len(W))
print(W[2].state_id)
W[2].show()

W[2].show(0)   # Hormuz
W[2].show(1)   # Hormuz
W[2].show(4)   # US/Allied Cohesion
W[2].show(6)   # US-Iran Tension

3
W2

World State : W2
-------------------------------------------------------
 0  Oil Price                          116.00
 1  Hormuz Oil Flow                     65.00
 2  India Gulf Oil Imports              88.00
 3  Iran Sanctions Enforcement         100.00
 4  US/Allied Cohesion                 100.00
 5  Iran Internal Cohesion             100.00
 6  US-Iran Tension                    108.00
 7  Regional Armed Conflict            108.00

World State : W2

Vector 0 : Oil Price
----------------------------------------------------------------------
Index       : 116
Description : International crude oil price level, represented by the Brent benchmark.
Direction   : Higher index means higher oil price.
Status      : Brent crude prices remain elevated and highly volatile because the Strait of Hormuz remains closed to commercial shipping, exports remain constrained, and regional conflict continues. The US suspension of offensive operations and offer of negotiations modestly reduce the 

In [ ]:
def resetWorld():
    del W[1:]
    E.clear()

    print("World reset to W0.")
# ------------------------
resetWorld()

World reset to W0.


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)